# 02 — CustomerPulse: Executive Business KPI Analysis & RFM Customer Segmentation

## Executive Summary
This notebook calculates core financial Business Analyst KPIs, performs RFM (Recency, Frequency, Monetary) Customer Segmentation, and models $ / ₹ Revenue at Risk.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append("../")
from src.feature_engineering import engineer_features
from src.business_analysis import BusinessAnalytics

proc_path = "../data/processed/customer_churn_cleaned.csv"
df_clean = pd.read_csv(proc_path)
df_fe = engineer_features(df_clean)

ba = BusinessAnalytics(df_fe)
kpis = ba.calculate_executive_kpis()

print("=========================================")
print("        BUSINESS ANALYTICS KPIS          ")
print("=========================================")
for k, v in kpis.items():
    print(f"• {k:<25}: {v:,.2f}" if isinstance(v, float) else f"• {k:<25}: {v:,}")


## 1. RFM Customer Segmentation & Risk Matrix

In [ ]:
df_risk = ba.calculate_revenue_at_risk()
df_seg = ba.assign_customer_segments()

segment_summary = df_seg.groupby("customer_segment").agg(
    customer_count=("customer_id", "count"),
    avg_churn_prob=("churn_probability", "mean"),
    total_revenue_at_risk=("revenue_at_risk", "sum")
).reset_index()

print("--- CUSTOMER SEGMENTATION SUMMARY ---")
print(segment_summary)

plt.figure(figsize=(9, 5))
sns.barplot(data=segment_summary, x="customer_segment", y="total_revenue_at_risk", palette="rocket")
plt.title("Total Revenue at Risk by Customer Segment ($)", fontsize=14, fontweight="bold")
plt.xlabel("Customer Segment")
plt.ylabel("Revenue at Risk ($)")
plt.xticks(rotation=15)
plt.show()


## 2. Top At-Risk High Value Customers

In [ ]:
top_at_risk = ba.get_top_at_risk_customers(10)
top_at_risk
